In [ ]:
import cv2
import mediapipe as mp
import os
import matplotlib.pyplot as plt 
import numpy as np
from scipy.signal import butter, lfilter,  filtfilt, savgol_filter
import pywt
from ultralytics import YOLO
from scipy.ndimage import zoom

# Função para criar o filtro passa-banda
def butter_bandpass(lowcut, highcut, fs, order=4):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return b, a

# Aplicação do filtro passa-banda

def apply_bandpass_filter_lfilter(data, lowcut, highcut, fs, order=4):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    filtered_data = lfilter(b, a, data)
    return filtered_data

def moving_average(signal, window_size):
    return np.convolve(signal, np.ones(window_size) / window_size, mode='valid')

# Remoção de baseline com ajuste de spline
def remove_baseline(signal, poly_order=2):
    x = np.arange(len(signal))
    coeffs = np.polyfit(x, signal, poly_order)  
    baseline = np.polyval(coeffs, x)     
    return signal - baseline  # Remove a baseline

def normalized_signal (signal):
    sig= np.array(signal) 
    signal_min = np.min(sig)
    signal_max = np.max(sig)
    normalized_signal = (sig - signal_min) / (signal_max - signal_min)
    return normalized_signal

model = YOLO("C:/Users/Douglas/Downloads/yolov8m_200e.pt")

# Inicializar FaceMesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, refine_landmarks=True)

# Vídeo de entrada
cap = cv2.VideoCapture(r"C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\cohface\cohface\40\1\data.avi") 
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# ROI e parâmetros
resize_dim = (115, 50)
amplification_factor_x = 0.7 
amplification_factor_y = 0.7 

pontos_referencia = [103, 67, 104, 109, 10, 338, 297, 332, 333, 299, 337, 151, 108, 69, 104, 105, 66, 107, 9, 336, 296, 334]
conexoes = [
    (103, 67), (67, 109), (109, 10), (10, 338),
    (338, 297), (297, 332), (103, 105),
    (105, 66), (66, 107), (107, 9), (9, 336), (336, 296), (296, 334), (334, 332)
]

# Sinais
green_signal, R_channel, G_channel, B_channel = [], [], [], []
frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(frame, device='cuda')
    boxes = results[0].boxes.xyxy.cpu().numpy()

    if len(boxes) > 0:
        # Pegamos a primeira detecção
        x1, y1, x2, y2 = boxes[0].astype(int)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(frame_width, x2), min(frame_height, y2)

        face_roi = frame[y1:y2, x1:x2]
        frame_rgb_roi = cv2.cvtColor(face_roi, cv2.COLOR_BGR2RGB)

        result = face_mesh.process(frame_rgb_roi)

        if result.multi_face_landmarks:
            for face_landmarks in result.multi_face_landmarks:
                pontos = []
                for idx in pontos_referencia:
                    lm = face_landmarks.landmark[idx]
                    x = int(lm.x * (x2 - x1)) + x1
                    y = int(lm.y * (y2 - y1)) + y1
                    pontos.append((x, y))

                if pontos:
                    xs, ys = [p[0] for p in pontos], [p[1] for p in pontos]
                    x_min, x_max = max(0, min(xs)), min(frame_width, max(xs))
                    y_min, y_max = max(0, min(ys)), min(frame_height, max(ys))

                    roi = frame[y_min:y_max, x_min:x_max]
                    if roi.size > 0:
                        resized_roi = cv2.resize(roi, resize_dim)

                        R_channel.append(np.mean(resized_roi[:, :, 2]))
                        G_channel.append(np.mean(resized_roi[:, :, 1]))
                        B_channel.append(np.mean(resized_roi[:, :, 0]))
                        green_signal.append(np.mean(resized_roi[:, :, 1]))
      
    frame_count += 1

# Aplicar o filtro passa-banda no sinal verde
lowcut = 0.7  
highcut = 4.0  

window_size = 10  
poly_order = 2    # Ordem do polinômio para a baseline

# Conversão para numpy arrays
R_channel = np.array(R_channel)
G_channel = np.array(G_channel)
B_channel = np.array(B_channel)

# Cálculo dos sinais POS
X_POS = G_channel - 0.5 * (R_channel + B_channel)
Y_POS = 3 * G_channel - 2 * R_channel - B_channel

X_POS = (X_POS - np.mean(X_POS)) / np.std(X_POS)
Y_POS = (Y_POS - np.mean(Y_POS)) / np.std(Y_POS)

# Projeção do sinal POS
S_POS = X_POS + Y_POS

filtered_signal_2 = apply_bandpass_filter_lfilter(S_POS, lowcut, highcut, fps)

smoothed_signal_2 = moving_average(filtered_signal_2, window_size)
baseline_removed_signal_2 = remove_baseline (smoothed_signal_2, poly_order)

IPPG2 = baseline_removed_signal_2
IPPG2_normalized = normalized_signal (IPPG2)

filtered_signal_1 = apply_bandpass_filter_lfilter(green_signal, lowcut, highcut, fps)
smoothed_signal_1 = moving_average(filtered_signal_1, window_size)
baseline_removed_signal_1 = remove_baseline(smoothed_signal_1, poly_order)

IPPG1 = baseline_removed_signal_1   
IPPG1_normalized = normalized_signal(IPPG1)

# Encontrando IPPG3

IPPG3 = (IPPG1 + IPPG2) / 2

IPPG3_normalized = normalized_signal (IPPG3)

plt.figure(figsize=(12, 6))
plt.plot(IPPG1_normalized, label="Sinal Verde Filtrado e Suavizado", linewidth=2)
plt.xlabel("Segundos")
plt.ylabel("Amplitude Normalizada")
plt.title("Sinal IPPG1")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(IPPG2_normalized, label="Sinal POS Metodo", linewidth=2)
plt.xlabel("Segundos")
plt.ylabel("Amplitude Normalizada")
plt.title("Sinal IPPG2")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(IPPG3_normalized, label="Média IPPG1 e IPPG2", linewidth=2)
plt.xlabel("Segundos")
plt.ylabel("Amplitude Normalizada")
plt.title("Sinal IPPG3")
plt.legend()
plt.grid()
plt.show()

# CODIGO PARA RECORTAR O SINAL IPP3 
total_frames = 1600   
duracao_total = 53      
frs = total_frames / duracao_total

#print(f"Tamanho do sinal IPPG3_normalized: {len(IPPG3_normalized)}")

frames = np.arange(0, len(IPPG3_normalized))  
tempo = frames / frs

inicio_frame = int(1 * frs)
fim_frame = int(10 * frs)
sinal_recortado = IPPG3_normalized[inicio_frame:fim_frame + 1]
tempo_recortado = tempo[inicio_frame:fim_frame + 1]


def plot_and_save_wavelet(time, data, wavelet, input_video_path):
    # Calcula a CWT
    widths = np.geomspace(1, 1024, num=75)
    cwtmatr, freqs = pywt.cwt(
        data, widths, wavelet, sampling_period=np.diff(time).mean()
    )
    cwtmatr = np.abs(cwtmatr[:-1, :-1])
    print(cwtmatr.shape, cwtmatr.min(), cwtmatr.max())

    # Diretório do vídeo de entrada
    video_dir = os.path.dirname(input_video_path)
    video_name = os.path.splitext(os.path.basename(input_video_path))[0]

    # Salvar a matriz CWT
    np.save(os.path.join(video_dir, f"{video_name}_cwt.npy"), cwtmatr)

    # Gerar e salvar a imagem do espectrograma
    fig, ax = plt.subplots(figsize=(6, 6))
    pcm = ax.pcolormesh(time, freqs, cwtmatr, cmap='viridis', shading='auto')
    ax.set_axis_off()
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

    output_img_path = os.path.join(video_dir, f"{video_name}_espectrograma.png")
    fig.savefig(output_img_path, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

    print(f"Arquivos salvos em: {video_dir}")
    return True

fs = 30

plot_and_save_wavelet(tempo_recortado, sinal_recortado, wavelet='cgau1',
                      input_video_path=r"C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\cohface\cohface\40\1\data.avi",
                      output_base_dir=r"C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\cohface\cohface\40\1")


In [ ]:
import cv2
import mediapipe as mp
import os
import matplotlib.pyplot as plt 
import numpy as np
from scipy.signal import butter, filtfilt
from scipy.signal import savgol_filter
import pywt
from ultralytics import YOLO
from scipy.ndimage import zoom

def process_video(input_video_path, output_base_dir):
    # Função para criar o filtro passa-banda
    def butter_bandpass(lowcut, highcut, fs, order=4):
        nyquist = 0.5 * fs
        low = lowcut / nyquist
        high = highcut / nyquist
        b, a = butter(order, [low, high], btype='band')
        return b, a

    # Aplicação do filtro passa-banda
    def apply_bandpass_filter(data, lowcut, highcut, fs, order=4):
        b, a = butter_bandpass(lowcut, highcut, fs, order=order)
        filtered_data = filtfilt(b, a, data)
        return filtered_data

    def moving_average(signal, window_size):
        return np.convolve(signal, np.ones(window_size) / window_size, mode='valid')

    # Remoção de baseline com ajuste de spline
    def remove_baseline(signal, poly_order=2):
        x = np.arange(len(signal))
        coeffs = np.polyfit(x, signal, poly_order)  
        baseline = np.polyval(coeffs, x)     
        return signal - baseline  # Remove a baseline

    def normalized_signal(signal):
        sig= np.array(signal) 
        signal_min = np.min(sig)
        signal_max = np.max(sig)
        normalized_signal = (sig - signal_min) / (signal_max - signal_min)
        return normalized_signal

    model = YOLO("C:/Users/Douglas/Downloads/yolov8m_200e.pt")

# Inicializar FaceMesh
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, refine_landmarks=True)

    # Vídeo de entrada
    cap = cv2.VideoCapture(input_video_path) 
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    # ROI e parâmetros
    resize_dim = (115, 50)      
    amplification_factor_x = 0.7
    amplification_factor_y = 0.7  

    pontos_referencia = [103, 67, 104, 109, 10, 338, 297, 332, 333, 299, 337, 151, 108, 69, 104, 105, 66, 107, 9, 336, 296, 334]
    conexoes = [
        (103, 67), (67, 109), (109, 10), (10, 338),
        (338, 297), (297, 332), (103, 105),
        (105, 66), (66, 107), (107, 9), (9, 336), (336, 296), (296, 334), (334, 332)
    ]

    # Sinais
    green_signal, R_channel, G_channel, B_channel = [], [], [], []
    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model.predict(frame, device='cuda')
        boxes = results[0].boxes.xyxy.cpu().numpy()

        if len(boxes) > 0:
            # Pegamos a primeira detecção
            x1, y1, x2, y2 = boxes[0].astype(int)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(frame_width, x2), min(frame_height, y2)

            face_roi = frame[y1:y2, x1:x2]
            frame_rgb_roi = cv2.cvtColor(face_roi, cv2.COLOR_BGR2RGB)

            result = face_mesh.process(frame_rgb_roi)

            if result.multi_face_landmarks:
                for face_landmarks in result.multi_face_landmarks:
                    pontos = []
                    for idx in pontos_referencia:
                        lm = face_landmarks.landmark[idx]
                        x = int(lm.x * (x2 - x1)) + x1
                        y = int(lm.y * (y2 - y1)) + y1
                        pontos.append((x, y))

                    if pontos:
                        xs, ys = [p[0] for p in pontos], [p[1] for p in pontos]
                        x_min, x_max = max(0, min(xs)), min(frame_width, max(xs))
                        y_min, y_max = max(0, min(ys)), min(frame_height, max(ys))

                        roi = frame[y_min:y_max, x_min:x_max]
                        if roi.size > 0:
                            resized_roi = cv2.resize(roi, resize_dim)

                            R_channel.append(np.mean(resized_roi[:, :, 2]))
                            G_channel.append(np.mean(resized_roi[:, :, 1]))
                            B_channel.append(np.mean(resized_roi[:, :, 0]))
                            green_signal.append(np.mean(resized_roi[:, :, 1]))
        
        frame_count += 1

    # Aplicar o filtro passa-banda no sinal verde
    lowcut = 0.7  
    highcut = 4.0  
    window_size = 10  
    poly_order = 2    # Ordem do polinômio para a baseline

    # Conversão para numpy arrays
    R_channel = np.array(R_channel)
    G_channel = np.array(G_channel)
    B_channel = np.array(B_channel)

    # Cálculo dos sinais POS
    X_POS = G_channel - 0.5 * (R_channel + B_channel)
    Y_POS = 3 * G_channel - 2 * R_channel - B_channel

    X_POS = (X_POS - np.mean(X_POS)) / np.std(X_POS)
    Y_POS = (Y_POS - np.mean(Y_POS)) / np.std(Y_POS)

    # Projeção do sinal POS
    S_POS = X_POS + Y_POS
    filtered_signal_2 = apply_bandpass_filter(S_POS, lowcut, highcut, fps)

    smoothed_signal_2 = moving_average(filtered_signal_2, window_size)
    baseline_removed_signal_2 = remove_baseline(smoothed_signal_2, poly_order)

    IPPG2 = baseline_removed_signal_2
    IPPG2_normalized = normalized_signal(IPPG2)

    filtered_signal_1 = apply_bandpass_filter(green_signal, lowcut, highcut, fps)
    smoothed_signal_1 = moving_average(filtered_signal_1, window_size)
    baseline_removed_signal_1 = remove_baseline(smoothed_signal_1, poly_order)

    IPPG1 = baseline_removed_signal_1   
    IPPG1_normalized = normalized_signal(IPPG1)

    # Encontrando IPPG3
    IPPG3 = (IPPG1_normalized + IPPG2_normalized) / 2
    IPPG3_normalized = normalized_signal(IPPG3)

    # Recortar o sinal IPP3 
    total_frames = 1200
    duracao_total = 60      
    fps = total_frames / duracao_total

    frames = np.arange(0, len(IPPG3_normalized))  
    tempo = frames / fps

    inicio_frame = int(1 * fps)
    fim_frame = int(10 * fps)
    sinal_recortado = IPPG3_normalized[inicio_frame:fim_frame + 1]
    tempo_recortado = tempo[inicio_frame:fim_frame + 1]

    # Gerar o espectrograma
    def plot_and_save_wavelet(time, data, wavelet, input_video_path):
        # Calcula a CWT
        widths = np.geomspace(1, 1024, num=75)
        cwtmatr, freqs = pywt.cwt(
            data, widths, wavelet, sampling_period=np.diff(time).mean()
        )
        cwtmatr = np.abs(cwtmatr[:-1, :-1])
        print(f"Forma original da CWT: {cwtmatr.shape}, min: {cwtmatr.min()}, max: {cwtmatr.max()}")


        # Diretório do vídeo de entrada
        video_dir = os.path.dirname(input_video_path)
        video_name = os.path.splitext(os.path.basename(input_video_path))[0]

        np.save(os.path.join(video_dir, f"{video_name}_cwt.npy"), cwtmatr)

        # Gerar e salvar a imagem do espectrograma redimensionado
        fig, ax = plt.subplots(figsize=(6, 6))
        pcm = ax.imshow(cwtmatr, cmap='viridis', aspect='auto')
        ax.set_axis_off()
        plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

        output_img_path = os.path.join(video_dir, f"{video_name}_espectrograma.png")
        fig.savefig(output_img_path, bbox_inches='tight', pad_inches=0)
        plt.close(fig)

        print(f"Arquivos salvos em: {video_dir}")
        return True

    plot_and_save_wavelet(tempo_recortado, sinal_recortado, wavelet='cgau1',
                        input_video_path= input_video_path)
    
    return True

input_video = r"c:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\cohface\cohface\37\0\data.avi"   # caminho do vídeo que você quer testar
output_dir = r"c:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\cohface\cohface\37\0\data.avi"       # ou deixe vazio se não usa essa pasta

process_video(input_video, output_dir)




0: 480x640 1 Face, 62.4ms
Speed: 7.1ms preprocess, 62.4ms inference, 1019.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 47.0ms
Speed: 2.1ms preprocess, 47.0ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 49.2ms
Speed: 1.6ms preprocess, 49.2ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 54.8ms
Speed: 1.3ms preprocess, 54.8ms inference, 3.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 51.9ms
Speed: 1.6ms preprocess, 51.9ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 39.1ms
Speed: 1.6ms preprocess, 39.1ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 12.7ms
Speed: 1.3ms preprocess, 12.7ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 10.2ms
Speed: 1.0ms preprocess, 10.2ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 4

In [2]:
import os
def main():
    base_dir = r"C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\cohface\cohface"
    for person_id in range(40, 41):
        for session_id in range(4):
            video_path = os.path.join(base_dir, str(person_id), str(session_id), "data.avi")
            if os.path.exists(video_path):
                print(f"Processando: {video_path}")
                try:
                    process_video(video_path, os.path.join(base_dir, str(person_id), str(session_id)))
                except Exception as e:
                    print(f"Erro ao processar {video_path}: {e}")
            else:
                print(f"Vídeo não encontrado: {video_path}")

if __name__ == "__main__":
    main()

Processando: C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\cohface\cohface\40\0\data.avi

0: 480x640 1 Face, 65.2ms
Speed: 6.0ms preprocess, 65.2ms inference, 157.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 55.2ms
Speed: 2.1ms preprocess, 55.2ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 53.5ms
Speed: 2.7ms preprocess, 53.5ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 49.3ms
Speed: 1.4ms preprocess, 49.3ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 39.3ms
Speed: 1.8ms preprocess, 39.3ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 41.2ms
Speed: 2.3ms preprocess, 41.2ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 41.0ms
Speed: 1.2ms preprocess, 41.0ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 

In [ ]:
import os
from pathlib import Path

# Diretório onde os vídeos estão armazenados
video_dir = 'C:/Users/Douglas/OneDrive - cefet-rj.br/Documentos/Material _ faculdade/Projeto IC/TESTE'
# Lista todos os arquivos .mp4 no diretório

video_files = list(Path(video_dir).glob('*.mp4'))

output_path = 'C:/Users/Douglas/OneDrive - cefet-rj.br/Documentos/Material _ faculdade/Projeto IC/CWT_TESTE'
output_erro = 'C:/Users/Douglas/OneDrive - cefet-rj.br/Documentos/Material _ faculdade/Projeto IC/videos com erro'
fallid_videos = []

for video_file in video_files:
    try:
        input_video_path = os.path.join(video_dir, video_file)
        # Ou, com pathlib: input_video_path = video_dir / video_file
        process_video(input_video_path, output_path)
        continue
    except:
        fallid_videos.append(video_file)
        video_name = os.path.splitext(os.path.basename(input_video_path))[0]
        output_dir = os.path.join(output_erro, video_name)
        os.makedirs(output_dir, exist_ok=True)
        output_path_erro = os.path.join(output_dir, f"{video_name}_videos_erro.png")
        continue

